In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_A1_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_A1_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_A1_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_A1_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 200 trials.
Best value (Accuracy): 0.7898
Best Hyperparameters:
  outer_lr: 0.0002324923814786689
  wd: 4.573104605499491e-05
  num_experts: 27
  MOE_top_k: 9
  MOE_gate_temperature: 0.5109681248250418
  MOE_aux_coeff: 0.1786685639031224
  MOE_ctx_out_dim: 64
  MOE_ctx_hidden_dim: 64
  MOE_dropout: 0.04999178679548349
  MOE_aux_loss_plcmt: outer
  ft_lr: 0.0014398667772455788
  label_smooth: 0.26508009683406153


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [ ]:
# FULL

#params_to_plot_ARCH = [
#    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
#]

params_to_plot_OPT = [
    "outer_lr", "wd", "label_smooth", "ft_lr"
]

#params_to_plot_MAML = [
#    "maml_inner_steps", "maml_alpha_init_eval", "maml_use_lslr", "use_lslr_at_eval", "use_maml_msl", "episodes_per_epoch_train"
#]

params_to_plot_MOE_CORE = [
    "num_experts", "MOE_top_k", "MOE_aux_coeff", "MOE_gate_temperature"
]

params_to_plot_MOE_AUX = [
    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout"
]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
#fig_slice = plot_slice(study, params=params_to_plot_ARCH)
#fig_slice.show()

In [12]:
#fig_slice = plot_slice(study, params=params_to_plot_MAML)
#fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_CORE)
fig_slice.show()

In [14]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

In [15]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
130,130,0.789792,0.3075,2026-04-19 23:43:02.109335,0 days 00:05:47.906600
186,186,0.789583,0.2525,2026-04-20 00:30:40.917296,0 days 00:05:06.563473
177,177,0.787708,0.3300,2026-04-20 00:22:02.900925,0 days 00:06:06.586190
187,187,0.786979,0.2950,2026-04-20 00:33:13.435101,0 days 00:05:46.903950
178,178,0.786042,0.2725,2026-04-20 00:22:03.082657,0 days 00:05:59.165962
142,142,0.781771,0.2675,2026-04-19 23:50:37.460787,0 days 00:05:49.165073
149,149,0.776667,0.2900,2026-04-19 23:53:40.806128,0 days 00:06:02.404118
102,102,0.774792,0.2500,2026-04-19 23:15:11.867973,0 days 00:04:46.710479
181,181,0.774688,0.2750,2026-04-20 00:26:37.939803,0 days 00:06:49.798698
182,182,0.774167,0.2825,2026-04-20 00:26:37.985793,0 days 00:05:50.845081


In [ ]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)